# Faster R-CNN 베이스라인


## 0. Colab 원격 실행환경과 Drive 연결

**목적:** VS Code에서 연결한 Colab 런타임의 GPU를 확인하고, 개인 Google Drive의 프로젝트 폴더에 접근한다.

VS Code에서 이 노트북의 커널을 Colab → Auto Connect로 연결한 뒤 실행한다. Drive 연결을 승인하면 원본 데이터와 수정 라벨 CSV를 읽고, 학습 결과를 runs/faster_rcnn/에 저장한다. 로컬 커널의 변수와 파일 경로는 Colab 런타임에 자동으로 전달되지 않는다.


In [ ]:
import sys
from pathlib import Path

import torch
from google.colab import drive

# VS Code Colab 확장으로 연결한 원격 런타임에서 Drive를 연결한다.
drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/코드잇 ;)/Project1_pill-object-detection"
)
DATA_DIR = DRIVE_PROJECT_DIR / "data"
TRAIN_IMAGE_DIR = DATA_DIR / "raw" / "train_images"
ANNOTATION_DIR = DATA_DIR / "raw" / "train_annotations"
SPLIT_PATH = DATA_DIR / "processed" / "split.csv"
FIXES_PATH = DATA_DIR / "processed" / "label_fixes.csv"
RUNS_ROOT = DRIVE_PROJECT_DIR / "runs" / "faster_rcnn"

print("Python:", sys.version.split()[0])
print("현재 작업 폴더:", Path.cwd())
print("Colab 런타임:", Path("/content").exists())
print("Drive 연결:", Path("/content/drive/MyDrive").exists())
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nDrive 프로젝트 경로:", DRIVE_PROJECT_DIR)
for label, target_path in [
    ("train_images", TRAIN_IMAGE_DIR),
    ("train_annotations", ANNOTATION_DIR),
    ("split.csv", SPLIT_PATH),
    ("수정 라벨 CSV", FIXES_PATH),
]:
    print(f"{label}: {'확인' if target_path.exists() else '아직 없음'} | {target_path}")
print("학습 결과 저장 위치:", RUNS_ROOT)


**진행 기준:** Colab GPU와 Drive 연결을 확인한다. train 이미지·JSON·split.csv·label_fixes.csv가 위 Drive 경로에 있어야 한다. Drive 업로드가 진행 중이면 업로드 완료 후 1.1에서 파일 수와 데이터 연결을 검증한다. 이 노트북은 로컬 원본 파일을 수정하지 않는다.


## 1. 학습 데이터 준비

**목적:** Drive의 원본 JSON에서 클래스와 라벨을 읽고, 팀에서 고정한 split.csv와 수정 라벨을 적용한 뒤 Faster R-CNN 입력 형식으로 변환한다.

원본 JSON·이미지는 변경하지 않는다. 수정 라벨은 메모리에서 반영하며, Drive의 split.csv는 읽기만 한다. 수정 라벨 CSV가 없거나 형식이 맞지 않으면 학습 데이터 준비를 중단한다.


### 1.1 경로와 실행 설정

Drive에 복사한 원본 이미지·JSON, 분할표, 수정 라벨 CSV 경로를 확인한다.


In [ ]:
from pathlib import Path
import ast
import csv
import json
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# 0번에서 연결한 개인 Drive 프로젝트 경로를 사용한다.
DATA_SOURCE = "Google Drive (Colab)"
SPLIT_SEED = 410  # 공유된 분할표의 기준 시드 기록용

required_paths = {
    "train_images": TRAIN_IMAGE_DIR,
    "train_annotations": ANNOTATION_DIR,
    "split.csv": SPLIT_PATH,
    "label_fixes.csv": FIXES_PATH,
}
missing = [f"{name}: {target}" for name, target in required_paths.items() if not target.exists()]
if missing:
    raise FileNotFoundError(
        "학습 입력 파일을 찾지 못했습니다. 아래 자료를 Drive 프로젝트 폴더의 "
        "data/raw 또는 data/processed에 복사한 뒤 다시 실행하세요.\n- "
        + "\n- ".join(missing)
    )

assert TRAIN_IMAGE_DIR.is_dir(), f"학습 이미지 폴더가 아닙니다: {TRAIN_IMAGE_DIR}"
assert ANNOTATION_DIR.is_dir(), f"JSON 폴더가 아닙니다: {ANNOTATION_DIR}"
assert SPLIT_PATH.is_file(), f"분할표 파일이 아닙니다: {SPLIT_PATH}"
assert FIXES_PATH.is_file(), f"수정 라벨 CSV 파일이 아닙니다: {FIXES_PATH}"

print("데이터 출처:", DATA_SOURCE)
print("학습 이미지:", TRAIN_IMAGE_DIR)
print("학습 JSON:", ANNOTATION_DIR)
print("고정 분할표:", SPLIT_PATH)
print("수정 라벨표:", FIXES_PATH)
print("학습 결과 저장 루트:", RUNS_ROOT)


### 1.2 원본 JSON 읽기와 클래스 번호표

이미지 파일명별로 annotation을 모은다. 원본 `category_id`를 오름차순으로 정렬해 클래스 번호를 부여하고, Faster R-CNN에서 사용할 번호(배경 0, 약 클래스 1~56)와 원본 ID 역매핑도 함께 만든다.

In [ ]:
def load_raw_data(annotation_dir):
    image_by_name = {}
    annotations = []
    json_paths = sorted(annotation_dir.rglob("*.json"))

    for json_path in json_paths:
        data = json.loads(json_path.read_text(encoding="utf-8"))
        for image_info in data.get("images", []):
            image_by_name[image_info["file_name"]] = image_info
        name_by_image_id = {
            image_info["id"]: image_info["file_name"]
            for image_info in data.get("images", [])
        }
        for ann in data.get("annotations", []):
            annotations.append({
                **ann,
                "file_name": name_by_image_id[ann["image_id"]],
                "category_id": int(ann["category_id"]),
                "bbox": list(map(float, ann["bbox"])),
            })
    return json_paths, image_by_name, annotations

json_paths, image_by_name, raw_annotations = load_raw_data(ANNOTATION_DIR)
raw_annotation_count = len(raw_annotations)
original_category_ids = sorted({ann["category_id"] for ann in raw_annotations})

# EDA/YOLO 번호(0~55)와 Faster R-CNN 번호(1~56)를 함께 관리한다.
class_map = pd.DataFrame({"category_id": original_category_ids})
class_map["class_index_0_based"] = range(len(class_map))
class_map["faster_rcnn_label"] = class_map["class_index_0_based"] + 1
category_to_model_label = dict(zip(class_map["category_id"], class_map["faster_rcnn_label"]))
model_label_to_category = {v: k for k, v in category_to_model_label.items()}

print(f"JSON {len(json_paths)}개 | 이미지 {len(image_by_name)}장 | 원본 bbox {raw_annotation_count}개")
print(f"클래스 수: {len(class_map)}")
display(class_map)

### 1.3 팀에서 고정한 train/val 분할 불러오기

Drive의 split.csv를 그대로 사용한다. 파일명·조합 정보가 원본 이미지 목록과 맞는지, train 185장 / val 47장인지, 같은 약 조합이 두 split에 섞이지 않았는지 검증한다. 이 노트북은 분할표를 새로 생성하거나 덮어쓰지 않는다.


In [ ]:
assert SPLIT_PATH.is_file(), f"Drive에서 split.csv를 찾지 못했습니다: {SPLIT_PATH}"
split_df = pd.read_csv(SPLIT_PATH)

required_split_columns = {"file_name", "combo", "split"}
assert required_split_columns.issubset(split_df.columns), (
    f"split.csv에 필요한 열이 없습니다. 필요: {sorted(required_split_columns)}, "
    f"실제: {list(split_df.columns)}"
)
assert split_df["file_name"].is_unique, "split.csv에 중복 file_name이 있습니다."
assert set(split_df["split"].dropna().unique()) == {"train", "val"}, (
    f"split 값은 train/val이어야 합니다: {split_df['split'].unique()}"
)

image_df = pd.DataFrame({"file_name": sorted(image_by_name)})
image_df["combo"] = image_df["file_name"].str.split("_").str[0]

raw_split_names = set(image_df["file_name"])
provided_split_names = set(split_df["file_name"].astype(str))
assert raw_split_names == provided_split_names, (
    "split.csv와 원본 JSON의 이미지 목록이 다릅니다. "
    f"JSON에만 있음 {len(raw_split_names - provided_split_names)}장, "
    f"split.csv에만 있음 {len(provided_split_names - raw_split_names)}장"
)

split_df["file_name"] = split_df["file_name"].astype(str)
split_df["combo"] = split_df["combo"].astype(str)
split_df["split"] = split_df["split"].astype(str)
expected_combo = image_df.set_index("file_name")["combo"]
provided_combo = split_df.set_index("file_name")["combo"]
assert provided_combo.to_dict() == expected_combo.to_dict(), (
    "split.csv의 combo 값이 이미지 파일명에서 계산한 조합 코드와 다릅니다."
)

image_df = image_df.merge(
    split_df[["file_name", "split"]],
    on="file_name",
    how="left",
    validate="one_to_one",
)
assert image_df["split"].notna().all(), "분할이 배정되지 않은 이미지가 있습니다."
assert image_df["split"].value_counts().to_dict() == {"train": 185, "val": 47}, (
    f"기대 분할은 train 185 / val 47입니다: {image_df['split'].value_counts().to_dict()}"
)
assert image_df.groupby("combo")["split"].nunique().max() == 1, (
    "같은 약 조합이 train과 val 양쪽으로 나뉘었습니다."
)

print("분할표:", SPLIT_PATH)
print(image_df["split"].value_counts().to_string())
print("고유 조합 수:", image_df["combo"].nunique())
print("같은 조합의 train/val 혼합: 없음")


### 1.4 수정 라벨 적용과 분할 연결

Drive의 label_fixes.csv는 file_name, category_id, action, bbox, area 열로 읽는다. add 8건을 추가하고, modify 13건은 파일명과 클래스 ID로 기존 bbox 하나와 연결한다. 이 가운데 bbox가 실제로 달라지는 9건만 bbox를 교체하며, bbox가 같은 4건은 area 단독 수정으로 보고 학습 입력에 반영하지 않는다. 최종 771개 bbox가 되고, 각 bbox는 기존 이미지의 고정 train/val 분할을 따른다.


In [ ]:
def parse_fix_bbox(value, row_number):
    try:
        bbox = [float(v) for v in ast.literal_eval(value)]
    except (ValueError, SyntaxError, TypeError) as exc:
        raise ValueError(f"CSV {row_number}행 bbox를 읽을 수 없습니다: {value!r}") from exc
    if len(bbox) != 4 or not np.isfinite(bbox).all():
        raise ValueError(f"CSV {row_number}행 bbox는 유한한 숫자 4개여야 합니다: {bbox}")
    if bbox[2] <= 0 or bbox[3] <= 0:
        raise ValueError(f"CSV {row_number}행 bbox의 너비/높이는 양수여야 합니다: {bbox}")
    return bbox

with FIXES_PATH.open("r", encoding="utf-8-sig", newline="") as fixes_file:
    fixes_reader = csv.DictReader(fixes_file)
    expected_fix_columns = {"file_name", "category_id", "action", "bbox", "area"}
    actual_fix_columns = set(fixes_reader.fieldnames or [])
    assert expected_fix_columns.issubset(actual_fix_columns), (
        f"label_fixes.csv 필수 열이 없습니다. 필요: {sorted(expected_fix_columns)}, "
        f"실제: {sorted(actual_fix_columns)}"
    )
    fixes = list(fixes_reader)

def apply_fixes(image_by_name, annotations, fixes):
    fixed = [dict(ann) for ann in annotations]
    next_ann_id = max(int(ann.get("id", 0)) for ann in fixed) + 1
    applied = Counter()

    for row_number, row in enumerate(fixes, start=2):
        file_name = row["file_name"].strip()
        category_id = int(row["category_id"])
        action = row["action"].strip().lower()
        bbox_after = parse_fix_bbox(row["bbox"], row_number)
        if file_name not in image_by_name:
            raise KeyError(f"CSV {row_number}행 이미지가 JSON에 없습니다: {file_name}")

        if action == "add":
            image_info = image_by_name[file_name]
            fixed.append({
                "id": next_ann_id,
                "image_id": image_info["id"],
                "file_name": file_name,
                "category_id": category_id,
                "bbox": bbox_after,
                "area": bbox_after[2] * bbox_after[3],
                "iscrowd": 0,
            })
            next_ann_id += 1
            applied["add"] += 1
        elif action == "modify":
            matches = [
                ann for ann in fixed
                if ann["file_name"] == file_name
                and int(ann["category_id"]) == category_id
            ]
            assert len(matches) == 1, (
                f"CSV {row_number}행은 파일명·클래스로 원본 라벨 1개와 "
                f"매칭되어야 합니다. 실제 {len(matches)}개: {file_name}, {category_id}"
            )
            old_bbox = [float(v) for v in matches[0]["bbox"]]
            if np.allclose(old_bbox, bbox_after, rtol=0, atol=1e-6):
                applied["area_only_skipped"] += 1
                continue
            matches[0]["bbox"] = bbox_after
            applied["bbox_modified"] += 1
        else:
            raise ValueError(f"CSV {row_number}행 action은 add/modify여야 합니다: {action!r}")

    return fixed, applied

fixed_annotations, applied_fixes = apply_fixes(image_by_name, raw_annotations, fixes)
fixed_df = pd.DataFrame(fixed_annotations)
fixed_df["split"] = fixed_df["file_name"].map(image_df.set_index("file_name")["split"])

assert sum(row["action"].strip().lower() == "add" for row in fixes) == 8
assert sum(row["action"].strip().lower() == "modify" for row in fixes) == 13
assert applied_fixes["add"] == 8
assert applied_fixes["bbox_modified"] == 9
assert applied_fixes["area_only_skipped"] == 4
assert len(fixed_annotations) == 771
assert fixed_df["split"].isna().sum() == 0
assert fixed_df["category_id"].nunique() == 56
assert set(fixed_df["category_id"]) == set(original_category_ids)

print(f"bbox 추가 {applied_fixes['add']}개 | bbox 수정 {applied_fixes['bbox_modified']}개 | area 단독 수정 건너뜀 {applied_fixes['area_only_skipped']}개")
print("수정 후 bbox:", len(fixed_annotations))
print("분할 미배정 bbox:", int(fixed_df["split"].isna().sum()))


### 1.5 bbox와 클래스 라벨 변환

각 bbox를 `[x, y, width, height]`에서 Faster R-CNN 형식인 `[x_min, y_min, x_max, y_max]`로 바꾼다. 원본 클래스 ID도 미리 만든 번호표를 통해 모델 라벨로 바꾼다.

In [ ]:
def xywh_to_xyxy(bbox):
    x, y, width, height = map(float, bbox)
    return [x, y, x + width, y + height]

annotations_by_image = defaultdict(list)
for ann in fixed_annotations:
    info = image_by_name[ann["file_name"]]
    x1, y1, x2, y2 = xywh_to_xyxy(ann["bbox"])
    assert x2 > x1 and y2 > y1, f"너비/높이가 0 이하인 bbox: {ann}"
    assert x1 >= 0 and y1 >= 0 and x2 <= info["width"] and y2 <= info["height"], (
        f"이미지 경계를 벗어난 bbox: {ann}"
    )
    annotations_by_image[ann["file_name"]].append({
        "bbox_xyxy": [x1, y1, x2, y2],
        "category_id": ann["category_id"],
        "model_label": category_to_model_label[ann["category_id"]],
    })

print("이미지별 라벨 연결 완료:", len(annotations_by_image), "장")
print("변환 bbox 수:", sum(len(v) for v in annotations_by_image.values()))
print("Faster R-CNN 클래스 라벨 범위: 1~", len(class_map), "(0은 배경)")

### 1.6 이미지와 bbox 시각화

train에서 객체 3개 또는 4개가 있는 이미지를 고정 시드로 골라 확인한다. 초록색 bbox는 수정 라벨을 반영한 뒤 모델에 전달할 박스다.

In [ ]:
def show_examples(image_df, annotations_by_image, object_counts=(3, 4)):
    rng = np.random.default_rng(SPLIT_SEED)
    train_names = image_df.loc[image_df["split"] == "train", "file_name"].tolist()
    selected = []
    for count in object_counts:
        candidates = [name for name in train_names
                      if len(annotations_by_image[name]) == count]
        if candidates:
            selected.append((count, rng.choice(candidates)))

    if not selected:
        print("조건에 맞는 이미지가 없습니다.")
        return

    fig, axes = plt.subplots(1, len(selected), figsize=(7 * len(selected), 8))
    if len(selected) == 1:
        axes = [axes]
    for ax, (count, file_name) in zip(axes, selected):
        image = Image.open(TRAIN_IMAGE_DIR / file_name).convert("RGB")
        draw = ImageDraw.Draw(image)
        for ann in annotations_by_image[file_name]:
            x1, y1, x2, y2 = ann["bbox_xyxy"]
            draw.rectangle((x1, y1, x2, y2), outline="lime", width=4)
            draw.text((x1, max(0, y1 - 18)), str(ann["category_id"]), fill="lime", stroke_width=1, stroke_fill="black")
        ax.imshow(image)
        ax.set_title(f"objects={count} | {file_name}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_examples(image_df, annotations_by_image)

### 1단계 확인

- 원본 bbox에 수정표를 반영해 총 771개가 되었는가?
- Drive의 고정 분할표와 원본 이미지가 모두 연결되었는가?
- train 185장 / val 47장이고, 미배정 bbox가 0개인가?
- 시각화에서 수정 bbox가 실제 알약을 감싸는가?

이 확인을 통과하면 Dataset/DataLoader와 짧은 Faster R-CNN 스모크 학습을 진행한다.


## 2. 파이프라인 작동 확인

**목적:** 준비한 이미지와 라벨이 Dataset/DataLoader를 거쳐 Faster R-CNN 학습과 검증 추론까지 오류 없이 연결되는지 짧게 확인한다.

이 단계는 점수 비교가 아니라 입력 형식, 손실 계산, 가중치 업데이트, 예측 결과 출력 흐름을 점검한다. 학습은 train 데이터 일부 배치만 사용한다.


### 2.1 Dataset과 DataLoader 구성

**목적:** 이미지와 수정된 bbox·클래스 라벨을 Faster R-CNN 입력 형식으로 전달하고, 데이터에서 실제 사용되는 라벨 클래스 수와 형식을 확인한다.


In [ ]:
import time
import random

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import to_tensor

# 스모크 테스트용 설정. 본 학습 설정과는 별도로 둔다.
SMOKE_SEED = 0
SMOKE_BATCH_SIZE = 1
SMOKE_MAX_STEPS = 1
SMOKE_MIN_SIZE = 320
SMOKE_MAX_SIZE = 512
VAL_PREVIEW_COUNT = 8
PREDICTION_SCORE_THRESHOLD = 0.05
MATCH_IOU_THRESHOLD = 0.50

random.seed(SMOKE_SEED)
np.random.seed(SMOKE_SEED)
torch.manual_seed(SMOKE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SMOKE_SEED)

class PillDetectionDataset(Dataset):
    def __init__(self, image_frame, image_dir, labels_by_image):
        self.frame = image_frame.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.labels_by_image = labels_by_image

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        file_name = self.frame.loc[index, "file_name"]
        image = Image.open(self.image_dir / file_name).convert("RGB")
        image_tensor = to_tensor(image)

        objects = self.labels_by_image[file_name]
        boxes = torch.tensor(
            [obj["bbox_xyxy"] for obj in objects], dtype=torch.float32
        ).reshape(-1, 4)
        labels = torch.tensor(
            [obj["model_label"] for obj in objects], dtype=torch.int64
        )
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor(index, dtype=torch.int64),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return image_tensor, target

def detection_collate_fn(batch):
    # 이미지마다 객체 수가 달라 기본 collate 대신 목록으로 묶는다.
    return tuple(zip(*batch))

train_frame = image_df.loc[image_df["split"] == "train", ["file_name"]].copy()
val_frame = image_df.loc[image_df["split"] == "val", ["file_name"]].copy()
train_dataset = PillDetectionDataset(train_frame, TRAIN_IMAGE_DIR, annotations_by_image)
val_dataset = PillDetectionDataset(val_frame, TRAIN_IMAGE_DIR, annotations_by_image)
train_loader = DataLoader(
    train_dataset, batch_size=SMOKE_BATCH_SIZE, shuffle=True,
    num_workers=0, collate_fn=detection_collate_fn,
)
val_loader = DataLoader(
    val_dataset, batch_size=1, shuffle=False,
    num_workers=0, collate_fn=detection_collate_fn,
)

sample_images, sample_targets = next(iter(train_loader))
assert len(sample_images) == len(sample_targets) == SMOKE_BATCH_SIZE
assert sample_images[0].ndim == 3 and sample_images[0].shape[0] == 3
assert sample_targets[0]["boxes"].shape[1] == 4
assert sample_targets[0]["labels"].dtype == torch.int64
assert (sample_targets[0]["labels"] > 0).all(), "0은 배경 클래스이므로 실제 약 라벨에 쓸 수 없습니다."

print(f"train Dataset/DataLoader: {len(train_dataset)}장 / {len(train_loader)} batch")
print(f"val Dataset/DataLoader: {len(val_dataset)}장 / {len(val_loader)} batch")
print("샘플 이미지 텐서:", tuple(sample_images[0].shape), sample_images[0].dtype)
print("샘플 target:", {key: tuple(value.shape) for key, value in sample_targets[0].items()})

# 데이터에 실제 등장하는 클래스 라벨 수를 split별로 집계한다.
split_model_labels = {}
split_box_counts = {}
for split_name in ("train", "val"):
    split_names = image_df.loc[image_df["split"] == split_name, "file_name"]
    split_objects = [obj for name in split_names for obj in annotations_by_image[name]]
    split_model_labels[split_name] = sorted({obj["model_label"] for obj in split_objects})
    split_box_counts[split_name] = len(split_objects)

print(f"수정 후 전체 객체 라벨: {len(fixed_annotations)}개")
print(f"train 객체/등장 클래스: {split_box_counts['train']}개 / {len(split_model_labels['train'])}종")
print(f"val 객체/등장 클래스: {split_box_counts['val']}개 / {len(split_model_labels['val'])}종")
print(f"모델에 전달하는 실제 클래스 라벨 범위: 1~{len(class_map)} (0은 배경)")
print("첫 샘플의 model_label:", sample_targets[0]["labels"].tolist())
print("train에 등장한 Faster R-CNN model_label:", split_model_labels["train"])
print("val에 등장한 Faster R-CNN model_label:", split_model_labels["val"])


**결과 확인:** train/val 이미지 수와 bbox·등장 클래스 수를 확인한다. Faster R-CNN 라벨은 1~56이며 0은 배경이다. 이미지 텐서는 `[채널, 높이, 너비]`, bbox는 `[객체 수, 4]` 형태여야 한다.


### 2.2 Faster R-CNN 모델 생성

**목적:** 데이터의 56개 약 클래스와 배경을 출력하도록 Faster R-CNN ResNet-50 FPN 모델을 만들고, 현재 커널에서 사용할 수 있는 장치를 확인한다.

사전학습 가중치는 불러오지 않는다. 따라서 이 모델의 짧은 학습·예측 결과는 성능을 뜻하지 않는다.


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = len(class_map) + 1  # 배경 1개 + 약 클래스 56개

model = fasterrcnn_resnet50_fpn(
    weights=None,
    weights_backbone=None,
    num_classes=NUM_CLASSES,
    min_size=SMOKE_MIN_SIZE,
    max_size=SMOKE_MAX_SIZE,
)
model.to(DEVICE)

print("사용 장치:", DEVICE)
print("모델 출력 클래스 수:", NUM_CLASSES, "(약 클래스 56 + 배경 1)")
print("Faster R-CNN:", type(model).__name__)


**결과 확인:** GPU 커널이면 `cuda`, 그 외에는 `cpu`가 출력된다. CPU에서도 코드 흐름은 확인할 수 있지만 스모크 학습이 오래 걸릴 수 있다. 본 학습은 GPU 환경에서 진행한다.


### 2.3 짧은 스모크 학습

**목적:** train 배치가 모델에 들어가 손실을 계산하고, 역전파와 optimizer 업데이트까지 수행되는지 확인한다.


In [ ]:
optimizer = torch.optim.SGD(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005,
)

first_parameter = next(parameter for parameter in model.parameters() if parameter.requires_grad)
parameter_before = first_parameter.detach().reshape(-1)[0].item()
smoke_losses = []
smoke_start = time.perf_counter()
model.train()

for step, (images, targets) in enumerate(train_loader):
    images = [image.to(DEVICE) for image in images]
    targets = [
        {key: value.to(DEVICE) for key, value in target.items()}
        for target in targets
    ]

    loss_dict = model(images, targets)
    total_loss = sum(loss for loss in loss_dict.values())
    assert torch.isfinite(total_loss), f"손실이 유한한 값이 아닙니다: {loss_dict}"

    optimizer.zero_grad(set_to_none=True)
    total_loss.backward()
    optimizer.step()
    smoke_losses.append(float(total_loss.detach().cpu()))

    print(f"step {step + 1}: " + ", ".join(
        f"{name}={float(value.detach().cpu()):.4f}"
        for name, value in loss_dict.items()
    ))
    if step + 1 >= SMOKE_MAX_STEPS:
        break

smoke_seconds = time.perf_counter() - smoke_start
parameter_after = first_parameter.detach().reshape(-1)[0].item()
parameter_changed = parameter_before != parameter_after
assert smoke_losses, "학습 배치가 실행되지 않았습니다."
assert parameter_changed, "optimizer 실행 후에도 확인한 가중치가 바뀌지 않았습니다."

print(f"스모크 학습 배치 수: {len(smoke_losses)}")
print(f"총 손실: {smoke_losses[-1]:.4f}")
print(f"학습 소요 시간: {smoke_seconds:.1f}초")
print("가중치 업데이트 확인:", parameter_changed)


**결과 확인:** 손실 항목이 유한한 숫자로 출력되고, 가중치 업데이트 확인이 `True`이면 학습 단계가 실행된 것이다. 한두 배치의 손실값으로 모델 성능을 판단하지 않는다.


### 2.4 검증 이미지 예측·시각화

**목적:** val 이미지에서 모델 예측을 받아 bbox 시각화까지 연결되는지 확인한다.

초록색 bbox는 정답, 빨간색 bbox는 모델 예측이다. 스모크 모델은 무작위 초기화 상태라 예측 점수가 낮을 수 있다. 시각화 흐름 확인을 위해 추론 단계에서는 낮은 점수 후보도 받아 이미지마다 상위 5개까지 표시한다. 빨간색 박스가 있다는 사실은 정답을 맞혔다는 뜻이 아니다.


In [ ]:
# 기본 score threshold에서 모두 제거될 수 있으므로, smoke 시각화에서는 낮은 점수 후보도 받는다.
# 이는 시각화 점검용 설정이며 본 학습의 예측 임계값으로 사용하지 않는다.
model.roi_heads.score_thresh = 0.0
val_predictions = []
model.eval()
with torch.inference_mode():
    for images, targets in val_loader:
        image = images[0].to(DEVICE)
        prediction = model([image])[0]
        val_predictions.append({
            "image": image.cpu(),
            "target": {key: value.cpu() for key, value in targets[0].items()},
            "prediction": {key: value.detach().cpu() for key, value in prediction.items()},
        })
        if len(val_predictions) >= VAL_PREVIEW_COUNT:
            break

assert val_predictions, "검증 이미지를 추론하지 못했습니다."
print(f"검증 추론 완료: {len(val_predictions)}장")
print("이미지별 추론 후보 수:", [len(item["prediction"]["boxes"]) for item in val_predictions])

# 정답/예측 bbox를 같은 이미지에 그려 육안 확인한다.
fig, axes = plt.subplots(len(val_predictions), 1, figsize=(9, 5 * len(val_predictions)))
if len(val_predictions) == 1:
    axes = [axes]
for ax, item in zip(axes, val_predictions):
    image = item["image"].permute(1, 2, 0).numpy()
    ax.imshow(image)
    draw_target = item["target"]
    draw_prediction = item["prediction"]

    for box, label in zip(draw_target["boxes"], draw_target["labels"]):
        x1, y1, x2, y2 = box.tolist()
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, max(0, y1-4), f"GT:{model_label_to_category[int(label)]}", color="lime", fontsize=8,
                bbox={"facecolor":"black", "alpha":0.6, "pad":1})

    top_indices = torch.argsort(draw_prediction["scores"], descending=True)[:5]
    for index in top_indices:
        box = draw_prediction["boxes"][index]
        label = draw_prediction["labels"][index]
        score = draw_prediction["scores"][index]
        x1, y1, x2, y2 = box.tolist()
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, y2, f"Pred:{model_label_to_category[int(label)]} {float(score):.2f}", color="red", fontsize=8,
                bbox={"facecolor":"white", "alpha":0.7, "pad":1})

    ax.set_title(f"green=GT / red=prediction | {len(draw_target['boxes'])} GT, "
                 f"top {len(top_indices)} prediction candidates")
    ax.axis("off")
plt.tight_layout()
plt.show()

print("검증 이미지 시각화 완료:", len(val_predictions), "장")


**결과 확인:** 초록색 정답 bbox와 빨간색 상위 예측 후보가 함께 표시되는지 확인한다. 빨간색 후보는 낮은 점수일 수 있으며, 이 시각화는 모델이 맞혔는지 평가하는 용도가 아니다. 스모크 모델의 결과로 성능 결론을 내리지 않는다.

**스모크 테스트 통과 기준:** Dataset/DataLoader 확인, 유한한 학습 손실, 가중치 업데이트, 검증 추론 및 bbox 시각화가 오류 없이 끝난다. W&B는 이 단계를 통과한 뒤 본 학습 전에 연결한다. 혼동행렬은 첫 베이스라인을 여러 시드로 학습한 뒤 동일한 val 분할과 평가 기준으로 비교하고, 주 성능 지표 mAP@[0.75:0.95]을 보완하는 분석으로 사용한다.


## 3. 첫 베이스라인 학습

**목적:** 수정 라벨과 고정 분할을 사용해 Faster R-CNN을 학습하고, seed별 검증 성능과 학습 속도를 비교한다. 데이터 분할 seed 410과 train/validation 데이터는 고정하고, 학습 seed만 0·1·2로 바꾼다.

seed 비교에는 **validation mAP@[0.75:0.95]**를 사용한다. Test에는 정답 라벨이 없으므로 seed 선택에 쓰지 않는다. seed 비교가 끝나면 validation 성능이 가장 높은 체크포인트를 골라 4번에서 test 예측 CSV를 만든다.

Colab 세션 중단에 대비해 seed는 한 번에 하나씩 실행한다. 첫 실행은 0이며, 완료 후 실행 목록만 1, 다음에는 2로 바꿔 같은 학습 셀을 다시 실행한다. 각 실행의 체크포인트와 지표는 Drive에 남고 W&B에는 별도 run으로 기록된다.


### 3.1 필요한 라이브러리 준비

W&B 기록과 mAP 계산을 위해 Colab 런타임에 필요한 패키지를 설치한다. 로컬 가상환경에는 설치하지 않는다.


In [ ]:
%pip install -q wandb torchmetrics pycocotools


### 3.2 W&B 인증

이 코드는 Colab 런타임에서 API 키를 숨김 입력으로 받는다. 키는 노트북 셀에 입력값으로 남지 않으며, Kaggle Secret이나 로컬 터미널 로그인과는 별도로 인증해야 한다.


In [ ]:
import getpass
import wandb

wandb_api_key = getpass.getpass("W&B API key (입력값은 표시되지 않습니다): ")
if not wandb_api_key:
    raise ValueError("W&B API key를 입력해야 합니다.")

wandb.login(key=wandb_api_key, relogin=True)
del wandb_api_key
print("W&B 인증 완료")


### 3.3 첫 학습 설정

| 설정 | 값 | 용도 |
|---|---:|---|
| 모델 | Faster R-CNN ResNet-50 FPN | torchvision 표준 객체 검출 모델 |
| 분할 seed | 410 고정 | 팀에서 정한 train 185 / val 47 분할 |
| 학습 seed | 0 | 첫 기준 실행. 이후 1~2을 추가해 변동 확인 가능 |
| 최대 epoch | 100 | 학습 상한 |
| 조기 종료 | 최소 20 epoch, patience 15, min_delta 0.001 | 검증 mAP가 의미 있게 개선되지 않을 때 종료 |
| batch size | 1 | 고해상도 이미지의 GPU 메모리 사용 고려 |
| SGD | lr 0.001, momentum 0.9, weight decay 0.0005 | 첫 비교용 초기값이며 최적값을 뜻하지 않음 |
| 입력 크기 | min 640, max 960 | 작은 알약의 세부 정보를 고려한 초기 설정 |
| 사전학습 가중치 | 사용 안 함 | 현재 합의한 무작위 초기화 기준선 |

이 값들은 첫 기준 실험을 시작하기 위한 설정이며 튜닝 결과가 아니다. 매 epoch마다 mAP@[0.75:0.95]를 기록하고, 최고 mAP의 best.pt와 마지막 완료 epoch의 last.pt를 따로 저장한다. 실제 실행 seed·설정·epoch 기록은 W&B와 Drive에 남긴다.


In [ ]:
import csv
import json
import random
import time
from datetime import datetime

from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# 고정된 train/validation 분할을 유지하고 학습 seed만 바꿔 비교한다.
TRAIN_SEEDS = [0, 1, 2]
MAX_EPOCHS = 100
EARLY_STOP_MIN_EPOCHS = 20
EARLY_STOP_PATIENCE = 15
EARLY_STOP_MIN_DELTA = 0.001
BASELINE_BATCH_SIZE = 1
BASELINE_LR = 0.001
BASELINE_MOMENTUM = 0.9
BASELINE_WEIGHT_DECAY = 0.0005
BASELINE_MIN_SIZE = 640
BASELINE_MAX_SIZE = 960
BASELINE_SCORE_THRESHOLD = 0.001
CHECKPOINT_EVERY_EPOCHS = 5
MAP_IOU_THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]
SEED_SUMMARY_PATH = RUNS_ROOT / "seed_comparison.csv"

print("비교할 학습 seed:", TRAIN_SEEDS)
print("고정 분할 seed:", SPLIT_SEED)
print("평가 지표: mAP@[0.75:0.95]")


### 3.4 학습 함수와 W&B 기록

학습 seed마다 모델·optimizer를 새로 만들고, 같은 train/validation 이미지와 같은 하이퍼파라미터를 사용한다. 각 seed의 최고 validation mAP, best epoch, 학습 시간, 체크포인트 경로를 Drive의 `seed_comparison.csv`에 누적한다.


In [ ]:
def set_training_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def save_checkpoint(path, epoch, model, optimizer, val_map, config):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_map_75_95": val_map,
        "config": config,
    }, path)


def evaluate_map_and_speed(model, loader):
    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=MAP_IOU_THRESHOLDS,
    )
    model.eval()
    inference_seconds = 0.0
    image_count = 0
    with torch.inference_mode():
        for images, targets in loader:
            images_device = [image.to(DEVICE) for image in images]
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            outputs = model(images_device)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            inference_seconds += time.perf_counter() - start
            image_count += len(images)
            metric.update(
                [{key: value.detach().cpu() for key, value in output.items()} for output in outputs],
                [{key: value.detach().cpu() for key, value in target.items()} for target in targets],
            )
    result = metric.compute()
    return float(result["map"].item()), 1000 * inference_seconds / max(image_count, 1)


def make_wandb_preview(model, loader, max_images=3, score_threshold=0.30):
    previews = []
    model.eval()
    with torch.inference_mode():
        for index, (images, targets) in enumerate(loader):
            if index >= max_images:
                break
            prediction = model([images[0].to(DEVICE)])[0]
            file_name = val_frame.iloc[index]["file_name"]
            canvas = Image.open(TRAIN_IMAGE_DIR / file_name).convert("RGB")
            draw = ImageDraw.Draw(canvas)
            for box, label in zip(targets[0]["boxes"], targets[0]["labels"]):
                x1, y1, x2, y2 = box.tolist()
                draw.rectangle((x1, y1, x2, y2), outline="lime", width=4)
                draw.text((x1, max(0, y1 - 18)), f"GT:{model_label_to_category[int(label)]}", fill="lime")
            keep = prediction["scores"].detach().cpu() >= score_threshold
            for idx in torch.where(keep)[0][:20]:
                x1, y1, x2, y2 = prediction["boxes"][idx].detach().cpu().tolist()
                label = int(prediction["labels"][idx].detach().cpu())
                score = float(prediction["scores"][idx].detach().cpu())
                draw.rectangle((x1, y1, x2, y2), outline="red", width=3)
                draw.text((x1, min(canvas.height - 16, y2)), f"Pred:{model_label_to_category[label]} {score:.2f}", fill="red")
            previews.append(wandb.Image(canvas, caption=file_name))
    return previews


def run_training_seed(train_seed):
    if train_seed not in TRAIN_SEEDS:
        raise ValueError(f"허용된 학습 seed가 아닙니다: {train_seed}")

    set_training_seed(train_seed)
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    run_name = (
        f"fasterrcnn_resnet50_seed{train_seed}_max{MAX_EPOCHS}_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    )
    run_dir = RUNS_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=False)
    metrics_path = run_dir / "metrics.csv"
    config_path = run_dir / "config.json"

    config = {
        "model": "torchvision_fasterrcnn_resnet50_fpn",
        "pretrained_weights": False,
        "split_seed": int(SPLIT_SEED),
        "train_seed": int(train_seed),
        "max_epochs": MAX_EPOCHS,
        "early_stopping": {
            "min_epochs": EARLY_STOP_MIN_EPOCHS,
            "patience": EARLY_STOP_PATIENCE,
            "min_delta": EARLY_STOP_MIN_DELTA,
            "monitor": "val_map_75_95",
        },
        "batch_size": BASELINE_BATCH_SIZE,
        "learning_rate": BASELINE_LR,
        "momentum": BASELINE_MOMENTUM,
        "weight_decay": BASELINE_WEIGHT_DECAY,
        "min_size": BASELINE_MIN_SIZE,
        "max_size": BASELINE_MAX_SIZE,
        "score_threshold": BASELINE_SCORE_THRESHOLD,
        "num_classes_including_background": NUM_CLASSES,
        "map_iou_thresholds": MAP_IOU_THRESHOLDS,
        "train_images": len(train_dataset),
        "val_images": len(val_dataset),
        "fixed_boxes": len(fixed_annotations),
    }
    config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

    generator = torch.Generator().manual_seed(train_seed)
    train_loader_for_run = DataLoader(
        train_dataset,
        batch_size=BASELINE_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        collate_fn=detection_collate_fn,
        generator=generator,
    )
    val_loader_for_run = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=detection_collate_fn,
    )

    model = fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None,
        num_classes=NUM_CLASSES,
        min_size=BASELINE_MIN_SIZE,
        max_size=BASELINE_MAX_SIZE,
    ).to(DEVICE)
    # 낮은 점수 후보도 mAP 평가에 포함하도록 confidence 필터를 낮게 둔다.
    model.roi_heads.score_thresh = BASELINE_SCORE_THRESHOLD
    optimizer = torch.optim.SGD(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=BASELINE_LR,
        momentum=BASELINE_MOMENTUM,
        weight_decay=BASELINE_WEIGHT_DECAY,
    )

    fieldnames = [
        "epoch", "train_loss", "val_map_75_95", "epoch_seconds",
        "val_inference_ms_per_image",
    ]
    best_map_75_95 = -1.0
    best_epoch = 0
    best_for_stopping = -1.0
    epochs_without_improvement = 0
    last_epoch = 0
    total_training_seconds = 0.0
    final_val_map = -1.0
    training_start = time.perf_counter()

    wandb_run = wandb.init(
        entity="codeit-prj1-team2",
        project="pill-object-detection",
        name=run_name,
        config=config,
        tags=["faster-rcnn", "baseline", f"seed-{train_seed}"],
    )
    wandb_run.define_metric("epoch")
    wandb_run.define_metric("train/*", step_metric="epoch")
    wandb_run.define_metric("val/*", step_metric="epoch")
    wandb_run.define_metric("time/*", step_metric="epoch")

    try:
        with metrics_path.open("w", newline="", encoding="utf-8") as metrics_file:
            metric_writer = csv.DictWriter(metrics_file, fieldnames=fieldnames)
            metric_writer.writeheader()
            for epoch in range(1, MAX_EPOCHS + 1):
                epoch_start = time.perf_counter()
                model.train()
                running_loss = 0.0
                batch_count = 0

                for images, targets in train_loader_for_run:
                    images = [image.to(DEVICE) for image in images]
                    targets = [
                        {key: value.to(DEVICE) for key, value in target.items()}
                        for target in targets
                    ]
                    loss_dict = model(images, targets)
                    total_loss = sum(loss for loss in loss_dict.values())
                    if not torch.isfinite(total_loss):
                        raise FloatingPointError(
                            f"seed {train_seed}, epoch {epoch}에서 유한하지 않은 loss: {loss_dict}"
                        )
                    optimizer.zero_grad(set_to_none=True)
                    total_loss.backward()
                    optimizer.step()
                    running_loss += float(total_loss.detach().cpu())
                    batch_count += 1

                train_loss = running_loss / max(batch_count, 1)
                final_val_map, inference_ms = evaluate_map_and_speed(model, val_loader_for_run)
                epoch_seconds = time.perf_counter() - epoch_start
                last_epoch = epoch
                row = {
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_map_75_95": final_val_map,
                    "epoch_seconds": epoch_seconds,
                    "val_inference_ms_per_image": inference_ms,
                }
                metric_writer.writerow(row)
                metrics_file.flush()
                wandb_run.log({
                    "epoch": epoch,
                    "train/loss": train_loss,
                    "val/map_75_95": final_val_map,
                    "time/epoch_seconds": epoch_seconds,
                    "time/val_inference_ms_per_image": inference_ms,
                })

                if final_val_map > best_map_75_95:
                    best_map_75_95 = final_val_map
                    best_epoch = epoch
                    save_checkpoint(
                        run_dir / "best.pt", epoch, model, optimizer,
                        final_val_map, config,
                    )
                if final_val_map > best_for_stopping + EARLY_STOP_MIN_DELTA:
                    best_for_stopping = final_val_map
                    epochs_without_improvement = 0
                else:
                    epochs_without_improvement += 1
                if epoch % CHECKPOINT_EVERY_EPOCHS == 0:
                    save_checkpoint(
                        run_dir / "last.pt", epoch, model, optimizer,
                        final_val_map, config,
                    )

                print(
                    f"seed {train_seed} | epoch {epoch:03d}/{MAX_EPOCHS} "
                    f"| loss={train_loss:.4f} | mAP@[.75:.95]={final_val_map:.4f} "
                    f"| epoch={epoch_seconds:.1f}s | val={inference_ms:.1f}ms/image "
                    f"| early-stop={epochs_without_improvement}/{EARLY_STOP_PATIENCE}"
                )
                if (
                    epoch >= EARLY_STOP_MIN_EPOCHS
                    and epochs_without_improvement >= EARLY_STOP_PATIENCE
                ):
                    print(
                        f"Early stopping: {EARLY_STOP_PATIENCE} epoch 동안 "
                        f"mAP 개선 폭이 {EARLY_STOP_MIN_DELTA} 미만입니다."
                    )
                    break

        previews = make_wandb_preview(model, val_loader_for_run)
        if previews:
            wandb_run.log({"val/prediction_examples": previews})
        total_training_seconds = time.perf_counter() - training_start
        wandb_run.summary["best_val/map_75_95"] = best_map_75_95
        wandb_run.summary["best_epoch"] = best_epoch
        wandb_run.summary["last_epoch"] = last_epoch
        wandb_run.summary["total_training_seconds"] = total_training_seconds
    finally:
        wandb_run.finish()

    save_checkpoint(
        run_dir / "last.pt", last_epoch, model, optimizer, final_val_map, config
    )
    result = {
        "train_seed": train_seed,
        "status": "complete",
        "best_val_map_75_95": best_map_75_95,
        "best_epoch": best_epoch,
        "last_epoch": last_epoch,
        "total_training_seconds": round(total_training_seconds, 2),
        "run_name": run_name,
        "run_dir": str(run_dir),
        "best_checkpoint": str(run_dir / "best.pt"),
    }
    print("학습 완료:", result)
    return result


def record_seed_result(result):
    """완료된 seed 결과를 Drive 비교표에 추가하거나 갱신한다."""
    summary_fields = [
        "train_seed", "status", "best_val_map_75_95", "best_epoch",
        "last_epoch", "total_training_seconds", "run_name",
        "run_dir", "best_checkpoint",
    ]
    summary_by_seed = {}
    if SEED_SUMMARY_PATH.exists():
        with SEED_SUMMARY_PATH.open(newline="", encoding="utf-8") as summary_file:
            for row in csv.DictReader(summary_file):
                summary_by_seed[int(row["train_seed"])] = row
    summary_by_seed[int(result["train_seed"])] = result
    with SEED_SUMMARY_PATH.open("w", newline="", encoding="utf-8") as summary_file:
        writer = csv.DictWriter(summary_file, fieldnames=summary_fields)
        writer.writeheader()
        writer.writerows(summary_by_seed[seed] for seed in sorted(summary_by_seed))
    comparison = pd.read_csv(SEED_SUMMARY_PATH).sort_values(
        "best_val_map_75_95", ascending=False
    )
    display(comparison)
    print("seed 비교표 저장:", SEED_SUMMARY_PATH)


### 3.5 학습 seed별 실행

아래 셀을 **seed 0 → seed 1 → seed 2 순서로 각각 실행**한다. 셀마다 학습 결과와 W&B run이 따로 생성되고, 완료 결과는 `seed_comparison.csv`에 누적된다. 세 셀을 한꺼번에 실행하는 `Run All`은 학습 시간이 길어질 수 있으니 사용하지 않는다.


#### 3.5.1 학습 seed 0


In [ ]:
seed_0_result = run_training_seed(0)
record_seed_result(seed_0_result)

#### 3.5.2 학습 seed 1

seed 0 셀이 완료된 뒤 실행한다.


In [ ]:
seed_1_result = run_training_seed(1)
record_seed_result(seed_1_result)

#### 3.5.3 학습 seed 2

seed 1 셀이 완료된 뒤 실행한다.


In [ ]:
seed_2_result = run_training_seed(2)
record_seed_result(seed_2_result)

## 4. Test 예측 및 Kaggle 제출 CSV 생성

**목적:** 3번에서 완료한 학습 seed 중 validation mAP@[0.75:0.95]가 가장 높은 run의 `best.pt`를 선택해 test 이미지를 예측하고, Kaggle 제출 형식으로 저장한다. Test 데이터는 모델 선택이나 성능 비교에 사용하지 않는다.

test 파일명 숫자를 `image_id`로 사용하며, Faster R-CNN 내부 클래스 번호(1~56)는 원본 `category_id`로 되돌린다. 예측 bbox는 `[x_min, y_min, x_max, y_max]`에서 Kaggle 형식의 `[x, y, width, height]`로 변환한다.


### 4.1 Validation 결과로 최종 체크포인트 선택

누적된 seed 비교표에서 최고 validation mAP를 기록한 완료 run을 자동 선택한다. 이 선택은 validation 결과만 기준으로 한다.


In [ ]:
if not SEED_SUMMARY_PATH.exists():
    raise FileNotFoundError("seed 비교표가 없습니다. 먼저 3번 학습을 완료하세요.")

seed_comparison = pd.read_csv(SEED_SUMMARY_PATH)
completed = seed_comparison.loc[seed_comparison["status"] == "complete"].copy()
if completed.empty:
    raise RuntimeError("완료된 학습 seed가 없습니다.")
best_run_row = completed.sort_values("best_val_map_75_95", ascending=False).iloc[0]
BEST_CHECKPOINT_PATH = Path(best_run_row["best_checkpoint"])
assert BEST_CHECKPOINT_PATH.exists(), f"체크포인트가 없습니다: {BEST_CHECKPOINT_PATH}"

print("선택한 학습 seed:", int(best_run_row["train_seed"]))
print("최고 validation mAP@[0.75:0.95]:", float(best_run_row["best_val_map_75_95"]))
print("체크포인트:", BEST_CHECKPOINT_PATH)
print("완료되어 비교 가능한 seed 수:", len(completed))

### 4.2 Test 이미지 예측 및 제출 파일 생성

모든 test 이미지를 추론한다. 제출 CSV는 `annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score` 순서로 저장한다.


In [ ]:
TEST_IMAGE_DIR = DATA_DIR / "raw" / "test_images"
if not TEST_IMAGE_DIR.exists():
    raise FileNotFoundError(f"test 이미지 폴더가 없습니다: {TEST_IMAGE_DIR}")

checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
checkpoint_config = checkpoint["config"]
submission_model = fasterrcnn_resnet50_fpn(
    weights=None,
    weights_backbone=None,
    num_classes=int(checkpoint_config["num_classes_including_background"]),
    min_size=int(checkpoint_config["min_size"]),
    max_size=int(checkpoint_config["max_size"]),
).to(DEVICE)
submission_model.load_state_dict(checkpoint["model_state_dict"])
submission_model.roi_heads.score_thresh = float(checkpoint_config.get("score_threshold", 0.001))
submission_model.eval()

image_extensions = {".png", ".jpg", ".jpeg", ".webp"}
test_image_paths = sorted(
    (path for path in TEST_IMAGE_DIR.iterdir() if path.is_file() and path.suffix.lower() in image_extensions),
    key=lambda path: (not path.stem.isdigit(), int(path.stem) if path.stem.isdigit() else path.name),
)
if not test_image_paths:
    raise FileNotFoundError(f"test 이미지가 없습니다: {TEST_IMAGE_DIR}")
if any(not path.stem.isdigit() for path in test_image_paths):
    raise ValueError("test 이미지 파일명이 숫자 image_id 형식인지 확인하세요. 숫자가 아닌 파일명: " + str([p.name for p in test_image_paths if not p.stem.isdigit()][:10]))
test_image_ids = [int(path.stem) for path in test_image_paths]
if len(test_image_ids) != len(set(test_image_ids)):
    raise ValueError("숫자 파일명에서 중복 image_id가 확인됐습니다.")

prediction_rows = []
start_time = time.perf_counter()
with torch.inference_mode():
    for image_index, image_path in enumerate(test_image_paths, start=1):
        with Image.open(image_path) as source_image:
            image = source_image.convert("RGB")
            width, height = image.size
            image_tensor = to_tensor(image).to(DEVICE)
        prediction = submission_model([image_tensor])[0]
        boxes = prediction["boxes"].detach().cpu()
        labels = prediction["labels"].detach().cpu()
        scores = prediction["scores"].detach().cpu()
        for box, model_label, score in zip(boxes, labels, scores):
            x1, y1, x2, y2 = map(float, box.tolist())
            x1 = min(max(x1, 0.0), float(width))
            y1 = min(max(y1, 0.0), float(height))
            x2 = min(max(x2, 0.0), float(width))
            y2 = min(max(y2, 0.0), float(height))
            box_width = x2 - x1
            box_height = y2 - y1
            if box_width <= 0 or box_height <= 0:
                continue
            model_label = int(model_label)
            if model_label not in model_label_to_category:
                raise ValueError(f"역매핑할 수 없는 모델 클래스 번호: {model_label}")
            prediction_rows.append({
                "image_id": int(image_path.stem),
                "category_id": int(model_label_to_category[model_label]),
                "bbox_x": x1,
                "bbox_y": y1,
                "bbox_w": box_width,
                "bbox_h": box_height,
                "score": float(score),
            })
        if image_index % 100 == 0 or image_index == len(test_image_paths):
            print(f"test 추론: {image_index}/{len(test_image_paths)}장")

submission_columns = [
    "annotation_id", "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score",
]
submission_df = pd.DataFrame(prediction_rows)
if submission_df.empty:
    raise RuntimeError("제출할 예측 박스가 없습니다. score threshold와 모델 출력을 확인하세요.")
submission_df.insert(0, "annotation_id", range(1, len(submission_df) + 1))
submission_df = submission_df[submission_columns]

assert submission_df[submission_columns].notna().all().all(), "제출표에 결측값이 있습니다."
assert submission_df["annotation_id"].is_unique, "annotation_id가 중복됐습니다."
assert submission_df["category_id"].isin(set(model_label_to_category.values())).all(), "원본 category_id 역매핑을 확인하세요."
assert (submission_df[["bbox_w", "bbox_h"]] > 0).all().all(), "너비/높이가 0 이하인 bbox가 있습니다."
assert submission_df["score"].between(0, 1).all(), "score 범위가 0~1을 벗어났습니다."

selected_run_dir = Path(best_run_row["run_dir"])
SUBMISSION_PATH = selected_run_dir / "submission.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)
elapsed = time.perf_counter() - start_time
print("test 이미지 수:", len(test_image_paths))
print("예측 bbox 수:", len(submission_df))
print("추론 시간(초):", round(elapsed, 1))
print("제출 CSV:", SUBMISSION_PATH)
display(submission_df.head(10))